In [7]:
"""
Airtel money pdf extraction
"""

import re, csv, pdfplumber
import pandas as pd

SKIP_LINES = ["Date & Time", "Details", "Credited", "Debited", "Balance",
              "Summary", "Transaction Type", "Amount(ZMW)", "Detailed Statement",
              "Need Help", "Balance statement"]

HEADERS = ["TXN_DATE", "VALUE_DATE", "DESCRIPTION", "MONEY_OUT_OR_IN", "BALANCE", "TXN_TYPE"]

PATTERN = re.compile(
    r"^(\d{2}/\d{2}/\d{2})\s+"
    r"(.+?)\s+"
    r"([\d,]+\.?\d*|--)\s+"
    r"([\d,]+\.?\d*|--)\s+"
    r"([\d,]+\.?\d+)$"
)

TIME_REF_PATTERN = re.compile(r"^\d{2}:\d{2}\s+(AM|PM)\s+\(.+\)$")


def format_date(raw):
    day, month, year = raw.split("/")
    return f"{int(day)}/{int(month)}/20{year}"


def clean_amount(value):
    return float(value.replace(",", ""))


def read_pdf(pdf_path):
    lines = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            lines.extend((page.extract_text() or "").splitlines())
    return lines


def get_metadata(lines):
    def find(label):
        for line in lines:
            if label in line:
                return line.split(label)[-1].strip()
        return ""
    return {
        "account_owner": lines[0].strip() if lines else "",
        "account_no":    find("Balance statement for the period").split()[-1],
        "currency":      "ZMW",
        "opening":       clean_amount(find("Opening Balance")),
        "closing":       clean_amount(find("Closing Balance")),
    }


def get_transactions(lines):
    transactions = []
    for line in lines:
        line  = line.strip()
        match = PATTERN.match(line)
        if match:
            date, desc, credited, debited, balance = match.groups()
            is_credit = credited != "--" and clean_amount(credited) > 0
            amount    = credited if is_credit else debited
            amount    = "0" if amount == "--" else amount
            transactions.append({
                "TXN_DATE":        format_date(date),
                "VALUE_DATE":      format_date(date),
                "DESCRIPTION":     desc.strip(),
                "MONEY_OUT_OR_IN": clean_amount(amount),
                "BALANCE":         clean_amount(balance),
                "TXN_TYPE":        "CR" if is_credit else "DR",
            })
        elif transactions and TIME_REF_PATTERN.match(line):
            transactions[-1]["DESCRIPTION"] += " " + line
        elif transactions and line and not any(s in line for s in SKIP_LINES):
            if not line[0].isdigit():
                transactions[-1]["DESCRIPTION"] += " " + line

    if not transactions:
        raise ValueError("No transactions found — check the PDF format.")

    df = pd.DataFrame(transactions)
    df["MONEY_OUT_OR_IN"] = pd.to_numeric(df["MONEY_OUT_OR_IN"], errors="coerce")
    df["BALANCE"]         = pd.to_numeric(df["BALANCE"],         errors="coerce")
    return df


def write_csv(df, info, output_path):
   metadata = [
    "AccountType:MOBILE MONEY",
    f"StatementStartDate:{df['TXN_DATE'].iloc[0]}",
    f"StatementEndDate:{df['TXN_DATE'].iloc[-1]}",
    f"AccountNo:{info['account_no']}",
    f"Currency:{info['currency']}",
    f"AccountOwner:{info['account_owner']}",
    f"Opening:{info['opening']}",
    f"Closing:{info['closing']}",
]
   with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([""] + HEADERS)
        for i, item in enumerate(metadata):
            writer.writerow([i, "", "", item, "", "", ""])
        writer.writerow([13] + HEADERS)
        for i, row in df.iterrows():
            writer.writerow([i + 14] + row.tolist())


def run(pdf_path, output_path):
    lines = read_pdf(pdf_path)
    df    = get_transactions(lines)
    info  = get_metadata(lines)
    write_csv(df, info, output_path)
    print(f"Done — {len(df)} transactions saved to {output_path}")


run("Airtel Statement.pdf", "Airtel.csv")

Done — 343 transactions saved to Airtel.csv
